# Stack v3 document lengths at 8k MSL

## tl;dr

- **19.32%** of rendered Stack v3 repository documents are estimated to exceed 8,192 GPT-OSS tokens (95% file-cluster bootstrap CI: **18.99%–19.65%**).
- That is approximately **30.29 million repositories** (95% CI: **29.75M–30.82M**) out of an estimated **156.80 million** source rows.
- This is the impact of a hypothetical per-repository hard cap. Standard Megatron `GPTDataset` does **not** discard those tails: long documents continue into later fixed-length training samples.


## Context & Methods

### Key Assumptions

- Population: Parquet rows under `processed_hf/stack-v3-train`; one row is one repository.
- Rendering: production `repo_to_text`, global renderer seed `1234`, and stable `(source_file, source_row_index)` seed derivation.
- Tokenization: production GPT-OSS tokenizer plus the `<|return|>` EOS token.
- Estimate: two-stage probability sample (512 uniformly sampled files, 128 uniformly sampled rows per file), weighted by sampled Parquet row counts.
- Uncertainty: 95% file-cluster bootstrap interval. This captures file-to-file variation and is more conservative than treating all sampled repos as independent.
- `length > 8192` means a repository would be affected by a hypothetical hard per-document cap. Standard Megatron GPT pretraining instead flattens documents into a token stream and constructs 8k samples, so it does not discard these document tails.


In [1]:
from __future__ import annotations

import hashlib
import json
import multiprocessing as mp
import os
import random
import sys
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq


INPUT_ROOT = Path("/cb/ml-cold/michaelw/codesign_pretrain/processed_hf/stack-v3-train")
TOKENIZER_JSON = Path("/workspace/ws/checkpoints/gpt-oss-20b-hf/tokenizer.json")
EOS_TOKEN = "<|return|>"
RENDERER_SEED = 1234
SAMPLING_SEED = 20260928
SAMPLED_FILES = 512
ROWS_PER_FILE = 128
WORKERS = 64
MSL = 8_192
BOOTSTRAP_REPLICATES = 2_000

assert INPUT_ROOT.is_dir(), INPUT_ROOT
assert TOKENIZER_JSON.is_file(), TOKENIZER_JSON
sys.path.insert(0, "/cb/ws/michaelw/Megatron-Bridge")
os.environ["TOKENIZERS_PARALLELISM"] = "false"


with TOKENIZER_JSON.open(encoding="utf-8") as input_file:
    tokenizer_payload = json.load(input_file)
assert tokenizer_payload["model"]["type"] == "BPE"
assert tokenizer_payload.get("normalizer") is None
assert "ByteLevel" in json.dumps(tokenizer_payload["pre_tokenizer"])

# ByteLevel maps each input byte to one Unicode code point before BPE. Therefore
# len(vocab_token) is the maximum number of source bytes one normal token can
# consume. Include added-token byte lengths as well.
MAX_TOKEN_BYTE_SPAN = max(
    max(len(token) for token in tokenizer_payload["model"]["vocab"]),
    max(len(token["content"].encode("utf-8")) for token in tokenizer_payload["added_tokens"]),
)
# EOS occupies one of the 8,192 positions. If the source content alone has more
# characters than 8,191 maximum-width tokens can cover, the rendered document
# is mathematically guaranteed to exceed 8,192 total tokens. UTF-8 bytes are
# always >= Python characters, and renderer markup only increases the length.
GUARANTEED_OVER_8K_CONTENT_CHARS = (MSL - 1) * MAX_TOKEN_BYTE_SPAN
print(f"Maximum tokenizer byte span: {MAX_TOKEN_BYTE_SPAN}")
print(f"Guaranteed-over-8k content-character cutoff: {GUARANTEED_OVER_8K_CONTENT_CHARS:,}")


Maximum tokenizer byte span: 128
Guaranteed-over-8k content-character cutoff: 1,048,448


## Data

Read every Parquet footer for the exact population size, then select the deterministic two-stage sample.

In [2]:
paths = sorted(INPUT_ROOT.rglob("*.parquet"))
sampling_rng = random.Random(SAMPLING_SEED)
sampled_file_indices = sorted(sampling_rng.sample(range(len(paths)), SAMPLED_FILES))
sampled_paths = [paths[index] for index in sampled_file_indices]


def read_num_rows(path: Path) -> int:
    return pq.ParquetFile(path).metadata.num_rows


with ThreadPoolExecutor(max_workers=96) as executor:
    sampled_row_counts = list(executor.map(read_num_rows, sampled_paths))

sampled_file_specs = [
    (str(path), path.relative_to(INPUT_ROOT).as_posix(), row_count)
    for path, row_count in zip(sampled_paths, sampled_row_counts)
]
population_file_count = len(paths)
estimated_population_rows = population_file_count / SAMPLED_FILES * sum(sampled_row_counts)

population_bootstrap_rng = np.random.default_rng(SAMPLING_SEED)
sampled_row_counts_array = np.asarray(sampled_row_counts, dtype=np.float64)
population_bootstrap = np.empty(BOOTSTRAP_REPLICATES)
for replicate in range(BOOTSTRAP_REPLICATES):
    indices = population_bootstrap_rng.integers(0, SAMPLED_FILES, size=SAMPLED_FILES)
    population_bootstrap[replicate] = (
        population_file_count / SAMPLED_FILES * sampled_row_counts_array[indices].sum()
    )

print(f"Parquet files: {population_file_count:,}")
print(
    f"Estimated repository rows: {estimated_population_rows:,.0f} "
    f"(95% CI {np.quantile(population_bootstrap, 0.025):,.0f}–"
    f"{np.quantile(population_bootstrap, 0.975):,.0f})"
)
print(
    f"Sample design: {SAMPLED_FILES:,} files x {ROWS_PER_FILE:,} rows = "
    f"{SAMPLED_FILES * ROWS_PER_FILE:,} repos"
)
print(
    f"Sampled rows/file: min={min(sampled_row_counts):,}, "
    f"median={int(np.median(sampled_row_counts)):,}, max={max(sampled_row_counts):,}"
)


Parquet files: 8,216
Estimated repository rows: 156,802,520 (95% CI 155,900,839–157,458,048)
Sample design: 512 files x 128 rows = 65,536 repos
Sampled rows/file: min=468, median=19,166, max=19,358


### Tokenize sampled repositories

Only the selected tiny Parquet row groups and the two needed columns (`repo_path`, `files`) are read. No rendered text or tokens are written to disk.


In [3]:
_worker_tokenizer = None


def initialize_worker() -> None:
    global _worker_tokenizer
    from tokenizers import Tokenizer
    from tokenizers.processors import TemplateProcessing

    tokenizer = Tokenizer.from_file(str(TOKENIZER_JSON))
    eos_id = tokenizer.token_to_id(EOS_TOKEN)
    if eos_id is None:
        raise RuntimeError(f"EOS token {EOS_TOKEN!r} is missing")
    tokenizer.post_processor = TemplateProcessing(
        single="$A <EOS>",
        special_tokens=[("<EOS>", eos_id)],
        pair=None,
    )
    _worker_tokenizer = tokenizer


def stable_sampling_seed(relative_path: str) -> int:
    key = f"{SAMPLING_SEED}\0{relative_path}".encode()
    return int.from_bytes(hashlib.blake2b(key, digest_size=8, person=b"stack-len").digest(), "little")


def analyze_file(spec: tuple[str, str, int]) -> dict[str, object]:
    from tokenization.stack_v3_repo_to_text_27_variants import repo_to_text, source_row_seed

    absolute_path_string, relative_path, num_rows = spec
    parquet_file = pq.ParquetFile(absolute_path_string)
    target_rows = sorted(random.Random(stable_sampling_seed(relative_path)).sample(range(num_rows), ROWS_PER_FILE))

    targets_by_row_group: dict[int, list[tuple[int, int]]] = {}
    target_position = 0
    row_group_start = 0
    for row_group_index in range(parquet_file.metadata.num_row_groups):
        row_group_rows = parquet_file.metadata.row_group(row_group_index).num_rows
        row_group_end = row_group_start + row_group_rows
        selected = []
        while target_position < len(target_rows) and target_rows[target_position] < row_group_end:
            absolute_row = target_rows[target_position]
            selected.append((absolute_row, absolute_row - row_group_start))
            target_position += 1
        if selected:
            targets_by_row_group[row_group_index] = selected
        row_group_start = row_group_end

    if _worker_tokenizer is None:
        raise RuntimeError("Worker tokenizer was not initialized")

    affected_rows = 0
    guaranteed_affected_rows = 0
    emitted_rows = 0
    exact_lengths = []
    for row_group_index in sorted(targets_by_row_group):
        table = parquet_file.read_row_group(row_group_index, columns=["repo_path", "files"])
        table_rows = table.to_pylist()
        exact_texts = []
        for absolute_row, local_row in targets_by_row_group[row_group_index]:
            row = table_rows[local_row]
            valid_contents = [
                item.get("content")
                for item in (row.get("files") or [])
                if isinstance(item, dict) and isinstance(item.get("content"), str)
            ]
            if not valid_contents:
                continue
            emitted_rows += 1
            content_characters = sum(len(content) for content in valid_contents)
            if content_characters > GUARANTEED_OVER_8K_CONTENT_CHARS:
                affected_rows += 1
                guaranteed_affected_rows += 1
                continue
            rendered = repo_to_text(
                row,
                seed=source_row_seed(RENDERER_SEED, relative_path, absolute_row),
            )
            exact_texts.append(rendered)

        encodings = _worker_tokenizer.encode_batch(exact_texts)
        group_lengths = [len(encoding.ids) for encoding in encodings]
        exact_lengths.extend(group_lengths)
        affected_rows += sum(length > MSL for length in group_lengths)
        del encodings, exact_texts, table_rows, table

    import gc
    import pyarrow as pa

    gc.collect()
    pa.default_memory_pool().release_unused()
    return {
        "relative_path": relative_path,
        "population_rows": num_rows,
        "sampled_rows": ROWS_PER_FILE,
        "emitted_rows": emitted_rows,
        "affected_rows": affected_rows,
        "guaranteed_affected_rows": guaranteed_affected_rows,
        "exact_max_tokens": max(exact_lengths, default=0),
    }


file_results = []
with ProcessPoolExecutor(
    max_workers=WORKERS,
    mp_context=mp.get_context("fork"),
    initializer=initialize_worker,
) as executor:
    futures = [executor.submit(analyze_file, spec) for spec in sampled_file_specs]
    for completed, future in enumerate(as_completed(futures), start=1):
        file_results.append(future.result())
        if completed % 64 == 0 or completed == len(futures):
            print(f"Completed {completed}/{len(futures)} sampled files")

sampled_emitted = sum(int(result["emitted_rows"]) for result in file_results)
sampled_affected = sum(int(result["affected_rows"]) for result in file_results)
sampled_guaranteed = sum(int(result["guaranteed_affected_rows"]) for result in file_results)
print(f"Emitted sampled repos: {sampled_emitted:,}")
print(f">8k sampled repos: {sampled_affected:,} ({sampled_guaranteed:,} proven by conservative size bound)")


Completed 64/512 sampled files


Completed 128/512 sampled files


Completed 192/512 sampled files


Completed 256/512 sampled files


Completed 320/512 sampled files


Completed 384/512 sampled files


Completed 448/512 sampled files


Completed 512/512 sampled files


Emitted sampled repos: 65,536
>8k sampled repos: 12,675 (366 proven by conservative size bound)


## Results

In [4]:
file_population_rows = np.array([int(result["population_rows"]) for result in file_results], dtype=np.float64)
file_emitted_totals = np.array(
    [
        int(result["population_rows"]) / int(result["sampled_rows"]) * int(result["emitted_rows"])
        for result in file_results
    ],
    dtype=np.float64,
)
file_affected_totals = np.array(
    [
        int(result["population_rows"]) / int(result["sampled_rows"]) * int(result["affected_rows"])
        for result in file_results
    ],
    dtype=np.float64,
)

estimated_emitted_repos = population_file_count / SAMPLED_FILES * file_emitted_totals.sum()
estimated_affected_repos = population_file_count / SAMPLED_FILES * file_affected_totals.sum()
estimated_affected_rate = estimated_affected_repos / estimated_emitted_repos

bootstrap_rng = np.random.default_rng(SAMPLING_SEED + MSL)
boot_counts = np.empty(BOOTSTRAP_REPLICATES)
boot_rates = np.empty(BOOTSTRAP_REPLICATES)
for replicate in range(BOOTSTRAP_REPLICATES):
    indices = bootstrap_rng.integers(0, SAMPLED_FILES, size=SAMPLED_FILES)
    boot_count = population_file_count / SAMPLED_FILES * file_affected_totals[indices].sum()
    boot_emitted = population_file_count / SAMPLED_FILES * file_emitted_totals[indices].sum()
    boot_counts[replicate] = boot_count
    boot_rates[replicate] = boot_count / boot_emitted

summary = {
    "population_parquet_files": population_file_count,
    "population_rows_estimate": int(round(estimated_population_rows)),
    "population_rows_ci_low": int(round(np.quantile(population_bootstrap, 0.025))),
    "population_rows_ci_high": int(round(np.quantile(population_bootstrap, 0.975))),
    "sampled_files": SAMPLED_FILES,
    "sampled_rows": SAMPLED_FILES * ROWS_PER_FILE,
    "sampled_emitted_rows": sampled_emitted,
    "sampled_affected_rows": sampled_affected,
    "sampled_guaranteed_affected_rows": sampled_guaranteed,
    "estimated_emitted_repos": int(round(estimated_emitted_repos)),
    "affected_repos_estimate": int(round(estimated_affected_repos)),
    "affected_repos_ci_low": int(round(np.quantile(boot_counts, 0.025))),
    "affected_repos_ci_high": int(round(np.quantile(boot_counts, 0.975))),
    "affected_repo_percent": 100 * estimated_affected_rate,
    "affected_percent_ci_low": 100 * np.quantile(boot_rates, 0.025),
    "affected_percent_ci_high": 100 * np.quantile(boot_rates, 0.975),
    "largest_exactly_tokenized_repo_tokens": max(int(result["exact_max_tokens"]) for result in file_results),
    "maximum_token_byte_span": MAX_TOKEN_BYTE_SPAN,
}

print(json.dumps(summary, indent=2))


{
  "population_parquet_files": 8216,
  "population_rows_estimate": 156802520,
  "population_rows_ci_low": 155900839,
  "population_rows_ci_high": 157458048,
  "sampled_files": 512,
  "sampled_rows": 65536,
  "sampled_emitted_rows": 65536,
  "sampled_affected_rows": 12675,
  "sampled_guaranteed_affected_rows": 366,
  "estimated_emitted_repos": 156802520,
  "affected_repos_estimate": 30292264,
  "affected_repos_ci_low": 29752421,
  "affected_repos_ci_high": 30823725,
  "affected_repo_percent": 19.318735381639314,
  "affected_percent_ci_low": 18.990753989899787,
  "affected_percent_ci_high": 19.648098288373923,
  "largest_exactly_tokenized_repo_tokens": 445612,
  "maximum_token_byte_span": 128
}


## Takeaways

- `affected_repo_percent` estimates the share of rendered Stack v3 repositories longer than 8,192 tokens; the count includes the production EOS token.
- Very large repositories are classified with a strict lower bound, not a characters-per-token approximation: with this ByteLevel BPE tokenizer, no token spans more than `maximum_token_byte_span` source bytes.
- Standard Megatron GPT pretraining does **not** hard-truncate these repository documents. `GPTDataset` flattens indexed documents and advances through long documents using offsets when constructing fixed-length 8k samples.
- Once Stack v3 tokenization finishes, this estimate can be replaced by an exact scan of `.idx` sequence lengths without reading `.bin` files.
